# Simulated LFC Spectrum

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [GitHub](https://github.com/oirlab/HISPEC_DRP).</span>

Simulate a mock laser frequency comb (LFC) on BSPEC fiber 2 using `LFCNU0` / `LFCDELNU` keywords


In [ ]:
from pathlib import Path

import numpy as np

OUT = Path("/path/to/hispec_simulated_lfc_example")
OUT.mkdir(parents=True, exist_ok=True)

SPECTROGRAPH, FIBER = "BSPEC", 2

print(f"OUT = {OUT}")
print(f"spectrograph={SPECTROGRAPH}  fiber={FIBER}")


## 1. Build the LFC spectrum and apply it as a rate map

In [ ]:
from hispecdrp.simulations.e2e_pipeline import build_lfc_line_list

line_list, lfc_meta = build_lfc_line_list(SPECTROGRAPH, fiber=FIBER)
print(lfc_meta)

# Raw 1-D LFC spectrum
lfc_wl = np.unique(
    np.concatenate([np.asarray(v, dtype=float) for v in line_list.values()])
)
lfc_wl.sort()
PHOTONS_PER_LINE = 1.0e7  # ph/s per mode; sets the rate-map flux scale only
lfc_line_rate = np.full(lfc_wl.shape, PHOTONS_PER_LINE)
n_modes_traces = sum(len(v) for v in line_list.values())
print(
    f"line-list spectrum: {lfc_wl.size} unique modes "
    f"({n_modes_traces} painted across all order traces), "
    f"{PHOTONS_PER_LINE:.1e} ph/s/line, "
    f"{lfc_wl.min():.4f}-{lfc_wl.max():.4f} um"
)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

from hispecdrp.simulations.e2e_pipeline import get_sim_hdf_trace_grids

# Rate-map rows outside this range are masked in the paint cell below.
Y_MIN, Y_MAX = 200, 3896
trace_grids = get_sim_hdf_trace_grids(SPECTROGRAPH)[FIBER]

SHOW_ORDERS = (110, 129, 148)
fig, axes = plt.subplots(len(SHOW_ORDERS), 1, figsize=(11, 6.5), sharex=False)
for ax, od in zip(axes, SHOW_ORDERS):
    spots = np.sort(line_list[int(od)])
    lo, hi = spots.min() - 0.0007, spots.max() + 0.0007
    injected = lfc_wl[(lfc_wl >= lo) & (lfc_wl <= hi)]
    grid = trace_grids[int(od)]
    gw = np.asarray(grid["wavelength"], dtype=float)
    gp = np.asarray(grid["pixel"], dtype=float)
    order_wl = np.argsort(gw)
    rows_inj = np.interp(injected, gw[order_wl], gp[order_wl])
    rows_spots = np.interp(spots, gw[order_wl], gp[order_wl])
    keep_inj = (rows_inj >= Y_MIN) & (rows_inj <= Y_MAX)
    keep_spots = (rows_spots >= Y_MIN) & (rows_spots <= Y_MAX)
    ax.vlines(
        injected[~keep_inj], 0.0, PHOTONS_PER_LINE,
        color="0.75", lw=0.7, label="masked (row outside Y_MIN/Y_MAX)",
    )
    ax.vlines(
        injected[keep_inj], 0.0, PHOTONS_PER_LINE, color="tab:blue", lw=0.7,
        label="injected line list",
    )
    ax.plot(
        spots[keep_spots],
        np.full(int(keep_spots.sum()), PHOTONS_PER_LINE), "|",
        ms=10, color="tab:orange",
        label=f"comb modes (n={int(keep_spots.sum())} of {spots.size})",
    )
    ax.set_xlim(lo, hi)
    ax.set_ylim(0, PHOTONS_PER_LINE * 1.25)
    ax.set_ylabel("photon rate\n(ph s$^{-1}$ line$^{-1}$)")
    ax.set_title(f"order {od}", loc="left", fontsize=10)
    ax.legend(fontsize=8, loc="upper right")
axes[-1].set_xlabel("wavelength (um)")
fig.suptitle("Raw 1-D LFC line-list spectrum before the HDF", fontsize=13)
fig.tight_layout()
fig.savefig(OUT / "lfc_1d_linelists_orders.png", dpi=130, bbox_inches="tight")
display(Image(filename=str(OUT / "lfc_1d_linelists_orders.png")))
plt.close(fig)


In [ ]:
from astropy import units as u

from hispecdrp.simulations import SimulateRateMap

srm = SimulateRateMap()
srm.set_fiber_spectrum(
    FIBER,
    lfc_wl * u.micron,
    lfc_line_rate * (u.ph / u.s),
    continuous=False,
)
rate = srm.create_native_rate_map(
    spectrograph=SPECTROGRAPH, method="prf_convolution",
    apply_blaze=False, save=False,
)

# No flux is expected outside this row range; mask the order ends.
rate[:Y_MIN] = 0
rate[Y_MAX:] = 0
print(
    f"HDF rate map: shape={rate.shape}, max={rate.max():.3e} ph/s/pix, "
    f"in-channel flux={rate[:, :4096].sum():.3e} ph/s"
)

In [ ]:
from hispecdrp.simulations.plotting import plot_lfc_rate_map_order_zooms

plot_lfc_rate_map_order_zooms(
    rate,
    line_list,
    spectrograph=SPECTROGRAPH,
    fiber=FIBER,
    out_path=OUT / "lfc_rate_map_order_zooms.png",
)


## 2. Extract, blaze, and run ProcessLFCPipeline

In [ ]:
import os

from hispecdrp import BlazeCorrectionPrimitive, datamodels
from hispecdrp.pipelines import ProcessLFCPipeline, Stage1Pipeline
from hispecdrp.simulations.e2e_pipeline import (
    create_tracepos_model_from_sim_hdf,
    create_traceprof_model_from_sim_hdf,
)
from hispecdrp.simulations.ideal_spectrum_rv import geometric_hdf_wavesol

os.environ["KOA_CALIBRATION_CACHE"] = str(OUT / "cals" / "lfc_local_caldb")

do_fibers = [str(FIBER)]
l0 = datamodels.L0Model(
    rate=np.asarray(rate, dtype=np.float32),
    meta={
        "instrument_name": "HISPEC",
        "spectrograph": SPECTROGRAPH,
        "datetime_obs": "2027-01-01T00:00:00.000",
        "exposure_type": "SPEC",
        "lfc_nu0": lfc_meta["nu0"],
        "lfc_delta_nu": lfc_meta["delta_nu"],
        "target_name": lfc_meta["fiber_object"],
    },
)
l0.rate_err = np.sqrt(np.maximum(l0.rate, 0.0)).astype(np.float32)
l0.rate_var_poisson = np.zeros_like(l0.rate, dtype=np.float32)
l0.rate_var_rnoise = np.zeros_like(l0.rate, dtype=np.float32)
l0.dq = np.zeros_like(l0.rate, dtype=np.uint32)

l1 = Stage1Pipeline(
    output_dir=OUT,
    dataset=l0,
    primitives={
        "spec_extract": {
            "do_fibers": do_fibers,
            "method": "box",
            "tracepos": create_tracepos_model_from_sim_hdf(SPECTROGRAPH),
            "traceprof": create_traceprof_model_from_sim_hdf(SPECTROGRAPH),
        },
        "set_wavesol": {"wavesol": geometric_hdf_wavesol(SPECTROGRAPH)},
    },
).apply()

n_orders, n_pix = getattr(l1, f"spec_uncal_fib{FIBER}").shape
fiberflat = datamodels.FiberflatModel(
    meta={"instrument_name": "HISPEC", "spectrograph": SPECTROGRAPH},
)
fiberflat[FIBER].spec = np.ones((n_orders, n_pix), dtype=np.float32)
fiberflat[FIBER].spec_err = np.full((n_orders, n_pix), 1.0e-2, dtype=np.float32)
fiberflat[FIBER].spec_dq = np.zeros((n_orders, n_pix), dtype=np.uint32)
l2 = BlazeCorrectionPrimitive.call(
    l1,
    do_fibers=do_fibers,
    fiberflat=fiberflat,
    output_dir=OUT,
    save_result=True,
)
l2.meta.lfc_nu0 = l0.meta.lfc_nu0
l2.meta.lfc_delta_nu = l0.meta.lfc_delta_nu

wavesol_model = ProcessLFCPipeline(
    output_dir=OUT,
    dataset=l2,
    primitives={
        "fit_lfc_lines": {"do_fibers": do_fibers, "fwhm_bounds": (1.0, 7.0)},
        "lfc_wavesol": {"do_fibers": do_fibers},
        "lsf": {"do_fibers": do_fibers},
    },
).apply()["wavesol"]
order_list = [int(o) for o in np.asarray(wavesol_model.make_order_vec())]
wavesol_path = OUT / "lfc_wavesol.fits"
wavesol_model.save(filepath=str(wavesol_path))
print(
    f"{type(l1).__name__} -> {type(l2).__name__} -> {type(wavesol_model).__name__}  "
    f"WAVE {getattr(wavesol_model, f'wave_fib{FIBER}').shape}  "
    f"fitted modes={len(getattr(wavesol_model, f'lfc_fitparams_fib{FIBER}'))}"
)
print(f"wrote {wavesol_path}")


### Order 118 fit

In [ ]:
from scipy.constants import c as C_LIGHT

import matplotlib.pyplot as plt

from hispecdrp.simulations import SimulateRateMap
from hispecdrp.simulations.simulate_rate_map import read_pyechelle_hdf

ORDER_SHOW = 118 if 118 in order_list else order_list[len(order_list) // 2]
oi = order_list.index(ORDER_SHOW)
pix = np.arange(4096, dtype=float)

# Pipeline WAVE for this order: line = WAVE row, knots = fitted LFC spots.
pipe_wave = np.asarray(getattr(wavesol_model, f"wave_fib{FIBER}"), dtype=float)[oi]
tab = getattr(wavesol_model, f"lfc_fitparams_fib{FIBER}")
sel = np.asarray(tab["ORDER"]) == ORDER_SHOW
lfc_pix = np.asarray(tab["MU_PIX"], dtype=float)[sel]
lfc_wave_knots = np.asarray(tab["WAVE"], dtype=float)[sel]
srt = np.argsort(lfc_pix)
lfc_pix, lfc_wave_knots = lfc_pix[srt], lfc_wave_knots[srt]

# HDF truth for this order: line = inverted PCHIP, knots = its 50 deposit samples.
srm = SimulateRateMap()
truth_wave = np.asarray(
    srm.get_wavelength_solution(SPECTROGRAPH)[FIBER][ORDER_SHOW]["wavelength"],
    dtype=float,
)
model_name = srm.bspec_hdf_model if SPECTROGRAPH == "BSPEC" else srm.rspec_hdf_model
model_data = read_pyechelle_hdf(srm.hdf_file_directory + model_name)
raw = model_data[f"fiber_{FIBER}"][ORDER_SHOW]
hdf_sample_wave = np.asarray(raw["xy_wavelengths"], dtype=float)
# get_wavelength_solution flips the spectral axis vs the HDF x coordinate.
hdf_sample_pix = 4095.0 - np.asarray(raw["x"], dtype=float)
srt = np.argsort(hdf_sample_pix)
hdf_sample_pix, hdf_sample_wave = hdf_sample_pix[srt], hdf_sample_wave[srt]

# Per-pixel velocity residual between the two lines.
ok = np.isfinite(pipe_wave) & np.isfinite(truth_wave)
dvel = np.where(ok, C_LIGHT * (pipe_wave - truth_wave) / truth_wave, np.nan)
# Residuals evaluated exactly at each knot set.
dvel_at_lfc = C_LIGHT * (lfc_wave_knots - np.interp(lfc_pix, pix, truth_wave)) / lfc_wave_knots
dvel_at_hdf = C_LIGHT * (np.interp(hdf_sample_pix, pix, pipe_wave) - hdf_sample_wave) / hdf_sample_wave

fig, axes = plt.subplots(
    2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [1.1, 1.0]}
)
ax = axes[0]
ax.plot(pix, truth_wave, color="tab:blue", lw=1.2,
        label="HDF: inverted PCHIP (get_wavelength_solution)")
ax.scatter(hdf_sample_pix, hdf_sample_wave, color="tab:blue", s=22, zorder=5,
           label=f"HDF deposit samples (n={hdf_sample_pix.size})")
ax.plot(pix, pipe_wave, color="tab:orange", lw=1.2,
        label="Pipeline WAVE (lfc_wavesol)")
ax.scatter(lfc_pix, lfc_wave_knots, color="tab:orange", s=14, zorder=5,
           label=f"Fitted LFC spots / spline knots (n={lfc_pix.size})")
ax.set_ylabel(r"$\lambda$ ($\mu$m)")
ax.set_title(f"Order {ORDER_SHOW}: two wavelength solutions, two knot sets")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.02), fontsize=8, ncol=1)

ax = axes[1]
for lo, hi in ((lfc_pix[0], lfc_pix[1]), (lfc_pix[-2], lfc_pix[-1])):
    ax.axvspan(lo, hi, color="tab:red", alpha=0.10)
ax.plot(pix, dvel, color="0.3", lw=0.8, label="Pipeline - HDF (per pixel)")
ax.scatter(hdf_sample_pix, dvel_at_hdf, color="tab:blue", s=22, zorder=5,
           label="residual at HDF sample positions")
ax.scatter(lfc_pix, dvel_at_lfc, color="tab:orange", s=16, zorder=5,
           label="residual at LFC spot positions")
ax.axhline(0.0, color="k", lw=0.8)
# Zoom on the bulk of the data: the Chebyshev WAVE runs away at the
# order edges (outside the fitted-knot range), which would otherwise
# set the scale.
core = dvel[(pix >= lfc_pix.min()) & (pix <= lfc_pix.max())]
y_lo, y_hi = np.nanpercentile(core, [0.5, 99.5])
y_pad = 0.2 * (y_hi - y_lo)
ax.set_ylim(y_lo - y_pad, y_hi + y_pad)
ax.set_xlabel("spectral pixel")
ax.set_ylabel(r"$\Delta v$ (m s$^{-1}$)")
ax.set_xlim(lfc_pix.min() - 100, lfc_pix.max() + 100)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.02), fontsize=8)
fig.tight_layout()
out_png = OUT / f"lfc_vs_hdf_order{ORDER_SHOW}_interpolation.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
# plt.show() is a no-op on non-interactive backends; display the PNG
# explicitly like the hispecdrp plotting helpers do.
from IPython.display import Image, display

display(Image(filename=str(out_png)))

print(
    f"order {ORDER_SHOW}: {lfc_pix.size} LFC knots "
    f"(spacing {np.median(np.diff(lfc_pix)):.1f} pix), "
    f"{hdf_sample_pix.size} HDF samples "
    f"(spacing {np.median(np.diff(hdf_sample_pix)):.1f} pix)"
)
print(
    f"median dv at LFC spots = {np.nanmedian(dvel_at_lfc):+.4f} m/s; "
    f"max |dv| between spots = {np.nanmax(np.abs(dvel)):.3f} m/s"
)


## 3. Derived wavesol vs `get_wavelength_solution`

In [ ]:
from hispecdrp.simulations import SimulateRateMap
from hispecdrp.simulations.plotting import plot_lfc_vs_hdf_wavesol_shift

lfc_wave = np.asarray(getattr(wavesol_model, f"wave_fib{FIBER}"), dtype=float)
flag = int(datamodels.DQ_FLAGS["UNRELIABLE_WS"])
dq = np.asarray(getattr(wavesol_model, f"spec_dq_fib{FIBER}"))
n_flagged = int(((dq & flag) != 0).sum())
lfc_wave = np.where((dq & flag) != 0, np.nan, lfc_wave)
print(f"masked UNRELIABLE_WS pixels: {n_flagged} of {dq.size}")
hdf_by_order = SimulateRateMap().get_wavelength_solution(SPECTROGRAPH)[FIBER]
hdf_wave = np.vstack(
    [np.asarray(hdf_by_order[int(od)]["wavelength"], dtype=float) for od in order_list]
)
stats = plot_lfc_vs_hdf_wavesol_shift(
    lfc_wave,
    hdf_wave,
    orders=order_list,
    resid_percentile=(1, 99),
    out_path=OUT / "lfc_vs_hdf_wavesol.png",
)
print(
    f"median dv = {stats['median_dvel_m_s']:.3f} m/s  "
    f"MAD = {stats['mad_dvel_m_s']:.3f} m/s  "
    f"clipped |dv|>100 m/s: {stats['n_clipped']}"
)
print(f"{'ord':>5} {'med_mA':>10} {'med_m/s':>10} {'MAD_m/s':>10}")
for row in stats["per_order"]:
    print(
        f"{row['order']:5d} {row['median_dlam_um']*1e7:10.3f} "
        f"{row['median_dvel_m_s']:10.3f} {row['mad_dvel_m_s']:10.3f}"
    )


In [ ]:
# Headline test: get_wavelength_solution at the fitted spot positions
# versus the exact comb wavelengths the pipeline assigned.
import matplotlib.pyplot as plt
from scipy.constants import c as C_LIGHT

tab = getattr(wavesol_model, f"lfc_fitparams_fib{FIBER}")
mu = np.asarray(tab["MU_PIX"], dtype=float)
wv = np.asarray(tab["WAVE"], dtype=float)
od = np.asarray(tab["ORDER"], dtype=int)
ok = np.isfinite(mu) & np.isfinite(wv)
mu, wv, od = mu[ok], wv[ok], od[ok]

lam_hdf_at_mu = np.empty_like(mu)
disp_at_mu = np.empty_like(mu)  # m/s per pixel at each spot
for o in order_list:
    sel = od == o
    row = hdf_wave[order_list.index(o)]
    lam_hdf_at_mu[sel] = np.interp(mu[sel], pix_grid := np.arange(row.size, dtype=float), row)
    disp_at_mu[sel] = C_LIGHT * np.abs(np.gradient(row))[
        np.clip(np.rint(mu[sel]).astype(int), 0, row.size - 1)
    ] / row[np.clip(np.rint(mu[sel]).astype(int), 0, row.size - 1)]

dv_spot = C_LIGHT * (lam_hdf_at_mu - wv) / wv
med = np.median(dv_spot)
mad = np.median(np.abs(dv_spot - med))
pix_off = dv_spot / disp_at_mu
print(f"fitted spots: {mu.size}")
print(
    f"at-spot dv (HDF - comb): median = {med:+.4f} m/s  "
    f"robust sigma = {1.4826 * mad:.4f} m/s"
)
print(
    f"equivalent spectral-pixel offset: median = {np.median(pix_off):+.3f} pix"
)

fig, ax = plt.subplots(figsize=(13, 3.6))
for o in order_list:
    sel = od == o
    ax.scatter(mu[sel], dv_spot[sel], s=3, label=f"order {o}" if o in (110, 148) else None)
ax.axhline(0.0, color="k", lw=0.9)
ax.set_xlabel("fitted spot pixel (MU_PIX)")
ax.set_ylabel(r"$\Delta v$ at spots (m s$^{-1}$)")
ax.set_title("get_wavelength_solution vs exact comb wavelength, at the spots")
fig.tight_layout()
out_png = OUT / "lfc_wavesol_at_spot_test.png"
fig.savefig(out_png, dpi=150)
plt.close(fig)
from IPython.display import Image, display

display(Image(filename=str(out_png)))


In [ ]:
plot_lfc_vs_hdf_wavesol_shift(
    lfc_wave,
    hdf_wave,
    orders=order_list,
    xlim=(800, 3500),
    resid_percentile=(1, 99),
    out_path=OUT / "lfc_vs_hdf_wavesol_center.png",
)


In [ ]:
from hispecdrp.simulations.plotting import plot_lfc_vs_hdf_wavesol_vs_wavelength

plot_lfc_vs_hdf_wavesol_vs_wavelength(
    lfc_wave,
    hdf_wave,
    orders=order_list,
    lam_xlim=(1.13, 1.17),
    out_path=OUT / "lfc_vs_hdf_wavesol_vs_lambda.png",
)
